# SOC Triage AI Agent — Experiment Notebook
**Author:** Rômulo Rocha · [linkedin.com/in/romrocha](https://linkedin.com/in/romrocha)

This notebook reproduces the full triage pipeline: ingest synthetic alerts → run the LangGraph ReAct agent → evaluate against ground truth → export a campaign report.

**Before running:**
1. Copy `.env.example` → `.env` and fill in your `OPENAI_API_KEY` and `MODEL_NAME`
2. Set `RESEARCH_ROUND` in your shell before launching JupyterLab: `export RESEARCH_ROUND=round1_no_ioc`
3. Run `pytest` from the project root to confirm all 70 tests pass

**Expected runtime:** ~25–30 min · ~$0.90 with `gpt-4o-mini` · ~100 alerts per round

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

print("Python:", sys.version.split()[0], "| exec:", sys.executable)

# Check critical dependencies — run `pip install -r requirements.txt` if any fail.
_REQUIRED = ["langchain_openai", "chromadb", "sentence_transformers", "langgraph"]
_missing = [m for m in _REQUIRED if not __import__("importlib").util.find_spec(m)]
if _missing:
    raise ImportError(f"Missing dependencies: {_missing}  →  pip install -r requirements.txt")

import importlib.metadata as _md
print("Stack:")
for _pkg in ("langchain", "langchain-openai", "langgraph",
             "langsmith", "chromadb", "sentence-transformers"):
    try:
        print(f"  {_pkg:24s} {_md.version(_pkg)}")
    except _md.PackageNotFoundError:
        print(f"  {_pkg:24s} (not installed)")


def _find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "security_agent" / "app" / "config.py").is_file():
            return p
    raise FileNotFoundError("security_agent/app/config.py not found above " + str(cur))


project_root = _find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root:", project_root)

env_path = project_root / ".env"
if env_path.is_file():
    load_dotenv(dotenv_path=env_path)
    print("Loaded .env:", env_path)
else:
    load_dotenv()
    print(f"Warning: {env_path} not found — falling back to environment variables")

# RESEARCH_ROUND controls which alert dataset and output folder the notebook uses.
# Set it in your shell before launching JupyterLab:
#   export RESEARCH_ROUND=round1_no_ioc   (options: round1_no_ioc · round2_no_ioc · round3_no_ioc)
# The three rounds use identical alert content with different time windows.
RESEARCH_ROUND = os.environ.get("RESEARCH_ROUND", "round1_no_ioc")
os.environ["RESEARCH_ROUND"] = RESEARCH_ROUND
print(f"RESEARCH_ROUND: {RESEARCH_ROUND}")

# Free label written to eval_runs.csv to distinguish experiment phases.
# Change this when you modify the dataset, prompts, or graph architecture.
EXPERIMENT_VERSION = "v1"


In [ ]:
# Cell 2 — Ingest alerts
# Clears any artifacts from a previous run, loads the round's JSON alerts into
# SQLite, and generates vector embeddings in ChromaDB.
#
# To reset between runs without re-running this cell:
#   rm -rf data/alerts.db data/chroma_db
#
# Expected output: "✓ N alerts ingested + embeddings in ChromaDB"

import importlib
import json
import shutil
from pathlib import Path

import security_agent.app.config as _app_cfg
importlib.reload(_app_cfg)
from security_agent.app.config import SQLITE_DB, CHROMA_DIR, ROUND_INPUT_DIR
from security_agent.app.ingestion.json_loader import ingest_alerts
from security_agent.app.ingestion.sqlite_store import SQLiteStore
from security_agent.app.ingestion.embedding import Embedder
from security_agent.app.ingestion.chroma_store import ChromaStore

if not ROUND_INPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"Alert directory not found: {ROUND_INPUT_DIR}\n"
        f"Check that RESEARCH_ROUND='{RESEARCH_ROUND}' matches a folder in input/"
    )

# ── Clear artifacts from previous run ─────────────────────────────────────
db_path = Path(SQLITE_DB)
chroma_path = Path(CHROMA_DIR)
removed = []
if db_path.parent.exists():
    for p in sorted(db_path.parent.glob(db_path.name + "*")):
        if p.is_file():
            p.unlink()
            removed.append(p.name)
if chroma_path.exists():
    shutil.rmtree(chroma_path)
    removed.append("chroma_db/")
db_path.parent.mkdir(parents=True, exist_ok=True)
print(f"Cleared: {', '.join(removed) if removed else 'nothing to clear'}")

# ── Load JSON alerts → SQLite ──────────────────────────────────────────────
print(f"Ingesting: {ROUND_INPUT_DIR.resolve()}")
store = ingest_alerts(directory=str(ROUND_INPUT_DIR))

sql = SQLiteStore(db_path=store.db_path)
cur = sql.conn.cursor()
cur.execute("SELECT alert_id, raw FROM alerts")
rows = cur.fetchall()
ids, texts = [], []
for r in rows:
    aid = r[0]
    raw = json.loads(r[1]) if isinstance(r[1], str) else r[1]
    text = (raw.get("title", "") + "\n" + raw.get("description", "")).strip() if isinstance(raw, dict) else str(raw)
    ids.append(aid)
    texts.append(text)

# ── Generate embeddings → ChromaDB ────────────────────────────────────────
# Uses all-MiniLM-L6-v2 (downloads ~90 MB on first run).
emb = Embedder()
chroma = ChromaStore(persist_directory=CHROMA_DIR)
EMBED_BATCH = 64
for i in range(0, len(ids), EMBED_BATCH):
    b_ids = ids[i : i + EMBED_BATCH]
    b_texts = texts[i : i + EMBED_BATCH]
    vectors = emb.encode(b_texts)
    chroma.upsert(
        ids=b_ids,
        embeddings=vectors,
        metadatas=[{"alert_id": a} for a in b_ids],
        documents=b_texts,
    )
print(f"✓ {len(ids)} alerts ingested + embeddings in ChromaDB")

# ── Reset alert statuses (unprocessed) and clear campaigns ────────────────
store_reset = SQLiteStore(db_path=SQLITE_DB)
store_reset.init_db()
n_alerts, n_campaigns = store_reset.reset_investigation_state()
stats = store_reset.get_alert_stats()
print(f"✓ {n_alerts} alerts reset, {n_campaigns} campaigns removed "
      f"(unprocessed={stats.get('unprocessed', 0)})")
print(f"✓ {RESEARCH_ROUND} ready to run")


## Cell 3–4 — Run the agent

The agent processes all alerts autonomously using this loop:

```
process_queue:
  select_seeds (1 alert/cycle)
    └─ run_investigation:
         investigate → correlate ⇄ investigate → decide → END
```

**What the agent does per investigation:**
- `investigate` — picks a seed alert and searches for related alerts by entity pivoting and vector similarity
- `correlate` — validates shared entities, time gaps, and alert patterns across candidates
- `decide` — writes the outcome to SQLite: create campaign, merge into existing campaign, or mark as false positive

**`MODEL_NAME`** (set in `.env`) is the only variable between experiments. Everything else — alert content, prompts, graph structure — stays constant.

**Live output** shows one line per investigation: elapsed time, seed IDs, campaigns created, false positives marked, and tools called.

In [ ]:
# Cell 4 — Execute the agent
import importlib
import logging
import os
import time
from collections import Counter
from datetime import datetime, timezone

# Reload bottom-up: sqlite_store → tools → nodes → langgraph_workflow.
# Skipping any module lets stale class definitions live in the kernel
# (e.g. SQLiteStore without the thread-local connection fix) and causes
# InterfaceError / AttributeError mid-run.
import security_agent.app.ingestion.sqlite_store as _store_mod
import security_agent.app.agent.tools as _tools_mod
import security_agent.app.agent.nodes as _nodes_mod
import security_agent.app.agent.langgraph_workflow as _wf_mod
importlib.reload(_store_mod)
importlib.reload(_tools_mod)
importlib.reload(_nodes_mod)
importlib.reload(_wf_mod)
from security_agent.app.agent import CampaignInvestigationWorkflow, setup_file_logging
from security_agent.app.config import CHROMA_DIR, ROUND_OUTPUT_DIR, SQLITE_DB
from security_agent.app.ingestion.sqlite_store import SQLiteStore

# ── Experiment parameters ──────────────────────────────────────────────────
# MODEL_NAME reads from .env. Override here only when comparing two models
# in the same session without editing .env.
MODEL_NAME = os.environ.get("MODEL_NAME", "gpt-4o-mini")
MAX_SEEDS = None        # None = process the entire alert queue
USE_LANGSMITH_TRACING = True
# ──────────────────────────────────────────────────────────────────────────

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M")
RUN_ID = f"{RESEARCH_ROUND}-{MODEL_NAME}-{RUN_TS}"

if USE_LANGSMITH_TRACING:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    # One stable LangSmith project per (round, model). Each execution adds
    # a separate trace inside it, named with the full RUN_ID.
    os.environ["LANGCHAIN_PROJECT"] = f"{RESEARCH_ROUND}-{MODEL_NAME}"
    print(f"LangSmith project: {os.environ['LANGCHAIN_PROJECT']!r}")

ROUND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ROUND_OUTPUT_DIR / f"{RESEARCH_ROUND}_agent_{MODEL_NAME}_{RUN_TS}.log"
log_handler = setup_file_logging(LOG_PATH)

store = SQLiteStore(db_path=SQLITE_DB)
store.init_db()
total_alerts = sum(store.get_alert_stats().values())
unprocessed_initial = store.get_alert_stats().get("unprocessed", 0)
print(f"RUN_ID : {RUN_ID}")
print(f"Model  : {MODEL_NAME}")
print(f"Alerts : {total_alerts} total · {unprocessed_initial} unprocessed")

workflow = CampaignInvestigationWorkflow(
    sqlite_path=str(SQLITE_DB),
    chroma_path=str(CHROMA_DIR),
    model_name=MODEL_NAME,
    temperature=0.0,
)

# Compact tool abbreviations for the live progress line.
_TOOL_ABBREV = {
    "search_similar_alerts":    "sim",
    "search_alerts_by_entity":  "ent",
    "fetch_alert_by_id":        "fetch",
    "fetch_campaign_alerts":    "case",
    "validate_shared_entities": "valid",
    "compute_time_delta":       "time",
    "ready_to_decide":          "ready",
    "create_campaign":          "+camp",
    "add_alerts_to_campaign":   "+to_camp",
    "mark_false_positive":      "+fp",
    "find_similar_cluster":     "cluster",
    "list_unprocessed_summary": "scan",
    "submit_selected_seeds":    "submit",
}


def _on_progress(info):
    inv = info["inv"]
    mx = info.get("max_seeds")
    inv_marker = f"{inv:>2d}/{mx}" if mx else f"{inv:>3d}"
    seeds_str = ",".join(s[:8] for s in info["seeds"])[:30]
    tools = info.get("tools_used") or {}
    tool_str = " ".join(
        f"{_TOOL_ABBREV.get(t, t[:8])}={c}"
        for t, c in sorted(tools.items(), key=lambda x: -x[1])[:6]
    )
    print(f"  [{inv_marker}] {info['elapsed_s']:5.1f}s | seeds: {seeds_str:30s} | "
          f"+{info['campaign']} camp  +{info['fp']} FP | {tool_str}")


print(f"\nStarting — this takes ~25–30 min for 100 alerts with gpt-4o-mini\n")
t0 = time.time()
results = workflow.process_queue(
    max_seeds=MAX_SEEDS,
    run_id=RUN_ID,
    on_progress=_on_progress,
    use_llm_seed_selection=True,
)
elapsed = time.time() - t0

logging.getLogger("soc_agent").removeHandler(log_handler)
log_handler.close()

print(f"\n{'='*60}")
print(f"Done in {elapsed:.1f}s ({elapsed/60:.1f} min) — {len(results)} result entries")
print(f"Log written to: {LOG_PATH}")
print(f"{'='*60}")

# Per-result summary — flags the queue-drain sentinel separately.
real_invs = 0
for r in results:
    cr = r.get("campaign_result") or {}
    action = cr.get("action", "?")
    if action == "mark_not_evaluated":
        ids = cr.get("alert_ids") or []
        print(f"  [drain] {len(ids)} alerts not evaluated (queue drain sentinel)")
        continue
    real_invs += 1
    tools = Counter()
    for t in cr.get("tool_outputs") or []:
        tools[t.get("tool", "?")] += 1
    tool_str = ", ".join(f"{k}={v}" for k, v in sorted(tools.items()))
    print(f"  [{real_invs:>2d}] {action} | {tool_str}")

print(f"\n{'='*60}")
print("Tool adoption across all investigations")
print(f"{'='*60}")
total_tools = Counter()
for r in results:
    for t in (r.get("campaign_result") or {}).get("tool_outputs") or []:
        total_tools[t.get("tool", "?")] += 1
for tool, n in total_tools.most_common():
    print(f"  {tool:30s}: {n:>3d}")


In [ ]:
# Cell 5 — Post-run inspection
# Prints alert status breakdown and lists detected campaigns from SQLite.
#
# What to look for:
#   - "not_evaluated" should be low (ideally 0). High values mean the agent
#     ran out of context or hit the max-iterations limit before deciding.
#   - Campaigns detected should match the expected count from ground truth (2).
#     More than 2 suggests fragmentation; fewer suggests under-clustering.

import json
from security_agent.app.config import SQLITE_DB
from security_agent.app.experiments.ground_truth import DATASET_FOLDER_SPECS
from security_agent.app.ingestion.sqlite_store import SQLiteStore

store = SQLiteStore(db_path=SQLITE_DB)
store.init_db()
cur = store.conn.cursor()

stats = store.get_alert_stats()
total = sum(stats.values())
not_evaluated = stats.get("not_evaluated", 0)
coverage = ((total - not_evaluated) / total * 100) if total else 0

print("Alert status breakdown:")
for status, count in sorted(stats.items()):
    print(f"  {status:20s}: {count:3d}  ({count/total*100:.0f}%)")
print(f"  {'TOTAL':20s}: {total:3d}")
print(f"\nLLM coverage: {coverage:.0f}%  ({total - not_evaluated}/{total} alerts acted on)")
if not_evaluated:
    print(f"  WARNING: {not_evaluated} alerts were not evaluated — check agent logs for context exhaustion")

cur.execute("""
    SELECT campaign_id, confidence, summary, alerts, run_id, created_at
    FROM campaigns ORDER BY created_at DESC
""")
campaigns = cur.fetchall()

expected_campaigns = sum(1 for _, label, _ in DATASET_FOLDER_SPECS if label == "in_campaign")
print(f"\nCampaigns detected: {len(campaigns)}  |  expected (ground truth): {expected_campaigns}")
if len(campaigns) > expected_campaigns:
    print(f"  Warning: {len(campaigns) - expected_campaigns} extra campaign(s) — possible fragmentation or FP-cluster")
elif len(campaigns) < expected_campaigns:
    print(f"  Warning: {expected_campaigns - len(campaigns)} missing campaign(s) — possible under-clustering")
else:
    print(f"  ✓ Campaign count matches ground truth")

for row in campaigns[:10]:
    aids = json.loads(row["alerts"]) if row["alerts"] else []
    n_alerts = len(aids)
    conf = float(row["confidence"] or 0)
    aids_preview = ", ".join(a[:8] for a in aids[:5])
    if len(aids) > 5:
        aids_preview += f", ... (+{len(aids) - 5})"
    print(f"\n  {row['campaign_id']}")
    print(f"    {n_alerts} alerts | confidence={conf:.0%} | run={row['run_id']}")
    print(f"    ids: [{aids_preview}]")
    print(f"    {(row['summary'] or '')[:120]}")


In [ ]:
# Cell 6 — Ground truth + evaluation
# Generates ground_truth.csv from the folder structure (if not already present)
# and scores the agent's SQLite decisions against it.
#
# Metrics explained:
#   Overall Accuracy   — correct labels / total alerts
#   Campaign Recall    — fraction of true campaign alerts correctly grouped
#   Campaign Precision — fraction of grouped alerts that are truly campaign
#   Purity             — single-campaign dominance within each detected cluster
#   Completeness       — correct campaign membership across all clusters
#   FP F1              — harmonic mean of false-positive precision and recall
#   Missed threats     — campaign alerts incorrectly marked as false positive (critical)

import importlib
from pathlib import Path
from IPython.display import display

import security_agent.app.config as _cfg
importlib.reload(_cfg)
from security_agent.app.config import ROOT, ROUND_INPUT_DIR, SQLITE_DB
from security_agent.app.experiments.eval import confusion_dataframes, evaluate_and_log, format_eval_report
from security_agent.app.experiments.ground_truth import write_ground_truth_csv

gt_path = write_ground_truth_csv(ROUND_INPUT_DIR.resolve())
print(f"✓ ground_truth.csv: {gt_path}")

eval_log = ROOT / "data" / "eval_runs.csv"
metrics = evaluate_and_log(
    gt_path,
    Path(SQLITE_DB),
    eval_log,
    model=MODEL_NAME,
    experiment_version=EXPERIMENT_VERSION,
)
print()
print(format_eval_report(metrics))

fp_conf, camp_conf = confusion_dataframes(metrics)
print("Confusion matrix — false positives")
display(fp_conf)
print("\nConfusion matrix — campaigns")
display(camp_conf)
print(f"\nRun history appended to: {eval_log}")


In [ ]:
# Cell 7 — Export campaign report
# Writes a human-readable report to output/<round>/ combining campaign details
# and evaluation metrics. Preview shows the first 40 lines inline.

from datetime import datetime, timezone
from pathlib import Path

from security_agent.app.config import GROUND_TRUTH_PATH, ROUND_OUTPUT_DIR, SQLITE_DB
from security_agent.app.experiments.report_export import build_campaign_report_body, build_full_report_with_eval
from security_agent.app.ingestion.sqlite_store import SQLiteStore

store = SQLiteStore(db_path=SQLITE_DB)
store.init_db()
gt_path = GROUND_TRUTH_PATH if GROUND_TRUTH_PATH.exists() else None

report_text = (
    build_full_report_with_eval(store, gt_path, Path(SQLITE_DB))
    if gt_path else build_campaign_report_body(store)
)

ROUND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_ts = RUN_TS if "RUN_TS" in globals() else datetime.now(timezone.utc).strftime("%Y%m%dT%H%M")
_label = MODEL_NAME.replace("/", "-") if "MODEL_NAME" in globals() else "unknown"
REPORT_PATH = ROUND_OUTPUT_DIR / f"{RESEARCH_ROUND}_results_{_label}_{_ts}.txt"
REPORT_PATH.write_text(report_text, encoding="utf-8")
print(f"Report saved: {REPORT_PATH}  ({len(report_text.splitlines())} lines)\n")

PREVIEW_LINES = 40
lines = report_text.splitlines()
print("\n".join(lines[:PREVIEW_LINES]))
if len(lines) > PREVIEW_LINES:
    print(f"\n... ({len(lines) - PREVIEW_LINES} more lines in file)")


In [ ]:
# Cell 8 — Evaluation history (optional)
# Loads eval_runs.csv and shows metrics across all runs side-by-side.
# Useful for comparing models or rounds after running the experiment multiple times.

import pandas as pd
from IPython.display import display
from security_agent.app.config import ROOT

eval_log = ROOT / "data" / "eval_runs.csv"
if not eval_log.exists():
    print(f"No evaluation history found at {eval_log}")
    print("Run Cell 6 to generate the first entry.")
else:
    df = pd.read_csv(eval_log)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.sort_values("timestamp", ascending=False)
    cols = ["timestamp", "experiment_version", "input_round", "model",
            "fp_f1", "camp_f1", "label_acc",
            "fp_precision", "fp_recall", "camp_precision", "camp_recall"]
    display(df[[c for c in cols if c in df.columns]].head(10))


In [ ]:
# Cell 9 — Cost and latency (optional)
# Combines local execution logs with LangSmith token data for the current round.
# Produces a per-run table and a per-model summary.
#
# Notes:
#   - Latency is parsed from the .log files in output/<round>/
#   - Token counts and cost are fetched from LangSmith (requires LANGCHAIN_API_KEY)
#   - If LangSmith is unavailable, only latency data is shown
#   - Results are persisted to data/cost_runs.csv for cross-session comparison

import os
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

from security_agent.app.config import ROOT

# Limit to the N most recent runs to avoid stale LangSmith lookups.
LATEST_N_RUNS = 3

# --- 1) Latency: parse execution logs for the current round ---------------
log_dir = ROOT / "output" / RESEARCH_ROUND
lat_rows = []
for log_path in sorted(log_dir.glob(f"{RESEARCH_ROUND}_agent_*.log")):
    try:
        last_line = log_path.read_text(encoding="utf-8").splitlines()[-1]
    except Exception:
        continue
    m = re.search(
        r"Total execution time:\s*([\d.]+)s.*?(\d+)\s+investigations.*?run_id=(\S+)",
        last_line,
    )
    if m:
        lat_rows.append({
            "run_id": m.group(3),
            "log_file": log_path.name,
            "total_seconds": float(m.group(1)),
            "investigations": int(m.group(2)),
        })

lat_df = pd.DataFrame(lat_rows)
if lat_df.empty:
    print(f"No execution logs found in {log_dir}")
    print("Run Cell 4 first to generate logs.")
    raise SystemExit(0)
print(f"✓ Latency: {len(lat_df)} run(s) found in output/{RESEARCH_ROUND}/")

if LATEST_N_RUNS is not None and len(lat_df) > LATEST_N_RUNS:
    lat_df["_ts_sort"] = lat_df["run_id"].str.extract(r"-(\d{8}T\d{4})$")[0]
    lat_df = (lat_df.sort_values("_ts_sort", ascending=False)
                    .head(LATEST_N_RUNS)
                    .drop(columns="_ts_sort")
                    .reset_index(drop=True))
    print(f"  → showing {LATEST_N_RUNS} most recent runs")


# --- 2) Token counts and cost from LangSmith ------------------------------
def _aggregate_run_metrics(client, rid: str) -> dict:
    """Sum tokens and cost for all LLM calls belonging to a specific run.

    Convention: LangSmith project = '{round}-{model}' (stable across runs).
    Individual runs are identified by a tag matching the full run_id.
    Falls back to using run_id as the project name for older setups.
    """
    totals = {"prompt_tokens": 0, "completion_tokens": 0,
              "total_cost": 0.0, "llm_calls": 0}
    parts = rid.rsplit("-", 1)
    project = parts[0] if len(parts) == 2 else rid
    fql = f'has(tags, "{rid}")'
    try:
        runs_iter = client.list_runs(project_name=project, run_type="llm", filter=fql)
    except Exception:
        runs_iter = client.list_runs(project_name=rid, run_type="llm")
    for r in runs_iter:
        totals["prompt_tokens"]     += int(getattr(r, "prompt_tokens", 0) or 0)
        totals["completion_tokens"] += int(getattr(r, "completion_tokens", 0) or 0)
        totals["total_cost"]        += float(getattr(r, "total_cost", 0.0) or 0.0)
        totals["llm_calls"]         += 1
    return totals


tok_df = pd.DataFrame(columns=["run_id", "prompt_tokens", "completion_tokens", "total_cost", "llm_calls"])
try:
    from langsmith import Client
    client = Client()

    rows = []
    for rid in lat_df["run_id"].tolist():
        try:
            metrics = _aggregate_run_metrics(client, rid)
            rows.append({"run_id": rid, **metrics})
            tag = "✓" if metrics["llm_calls"] > 0 else "∅"
            print(f"  {tag} {rid}: {metrics['llm_calls']} LLM calls, "
                  f"{metrics['prompt_tokens'] + metrics['completion_tokens']} tokens, "
                  f"${metrics['total_cost']:.4f}")
        except Exception as exc:
            print(f"  ✗ {rid}: {exc.__class__.__name__}: {exc}")
    tok_df = pd.DataFrame(rows)
except ImportError:
    print("LangSmith SDK not installed — showing latency only")
except Exception as exc:
    print(f"LangSmith unavailable ({exc.__class__.__name__}: {exc}) — showing latency only")

# --- 3) Merge and derive summary columns ----------------------------------
df = lat_df.merge(tok_df, on="run_id", how="left")
df[["_round", "_model", "_ts"]] = df["run_id"].str.extract(
    r"^(round\d+[^-]*)-(.+?)-(\d{8}T\d{4})$"
)
for col in ("prompt_tokens", "completion_tokens", "total_cost", "llm_calls"):
    if col not in df.columns:
        df[col] = 0
df["total_tokens"] = df["prompt_tokens"].fillna(0) + df["completion_tokens"].fillna(0)
df["minutes"] = (df["total_seconds"] / 60).round(1)
df["sec_per_investigation"]  = (df["total_seconds"] / df["investigations"]).round(1)
df["cost_per_investigation"] = (df["total_cost"]    / df["investigations"]).round(4)

# --- 4) Per-run table -----------------------------------------------------
print(f"\n=== {RESEARCH_ROUND} — per run (most recent first) ===")
per_run_cols = [
    "_ts", "_model", "investigations", "minutes", "sec_per_investigation",
    "llm_calls", "total_tokens", "total_cost", "cost_per_investigation",
]
display(df[per_run_cols].sort_values("_ts", ascending=False).reset_index(drop=True))

# --- 5) Averages by model -------------------------------------------------
print(f"\n=== {RESEARCH_ROUND} — averages by model ===")
summary = (
    df.groupby("_model")
      .agg(runs=("run_id", "count"),
           avg_minutes=("minutes", "mean"),
           avg_sec_per_inv=("sec_per_investigation", "mean"),
           avg_total_tokens=("total_tokens", "mean"),
           avg_cost_usd=("total_cost", "mean"),
           avg_cost_per_inv=("cost_per_investigation", "mean"))
      .round(3)
)
display(summary)

# --- 6) Persist to cost_runs.csv (cumulative, deduped by run_id) ----------
# Anti-regression rule: if LangSmith returned 0 LLM calls (unavailable or
# project expired), do NOT overwrite an existing row that has valid data.
# A new row with zero calls is only written if the run_id is not yet in the CSV.
cost_log = ROOT / "data" / "cost_runs.csv"
cost_cols = [
    "run_id", "_round", "_model", "_ts",
    "investigations", "total_seconds", "minutes", "sec_per_investigation",
    "llm_calls", "prompt_tokens", "completion_tokens", "total_tokens",
    "total_cost", "cost_per_investigation",
]
to_save = df.reindex(columns=cost_cols).copy()

if cost_log.exists():
    existing = pd.read_csv(cost_log)
    existing_ids = set(existing["run_id"])

    has_data = to_save["llm_calls"].fillna(0) > 0
    overwrites = to_save[has_data]
    additions  = to_save[~has_data & ~to_save["run_id"].isin(existing_ids)]
    skipped    = to_save[~has_data &  to_save["run_id"].isin(existing_ids)]

    combined = pd.concat([existing, overwrites, additions], ignore_index=True)
    combined = combined.drop_duplicates(subset=["run_id"], keep="last")
else:
    overwrites, additions, skipped = to_save, to_save.iloc[0:0], to_save.iloc[0:0]
    combined = to_save

combined = combined.sort_values("_ts", ascending=False).reset_index(drop=True)
cost_log.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(cost_log, index=False)
print(f"\n✓ Saved to {cost_log}")
print(f"  - {len(overwrites)} run(s) with LangSmith data (overwrite)")
print(f"  - {len(additions)} new run(s) without LangSmith data (written as zero)")
if len(skipped):
    print(f"  - {len(skipped)} run(s) with 0 calls skipped to preserve existing row: {list(skipped['run_id'])}")
print(f"  - total rows in history: {len(combined)}")
